In [ ]:
import pickle
import pandas as pd
import re
from string import punctuation
from nltk.corpus import wordnet, stopwords
from nltk.tokenize import word_tokenize
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from sklearn.model_selection import train_test_split


In [12]:
df = pd.read_csv('Symptom2Disease.csv')
df.head()    

,Unnamed: 0,label,text
0,0,Psoriasis,I have been experiencing a skin rash on my arm...
1,1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,2,Psoriasis,I have been experiencing joint pain in my fing...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,4,Psoriasis,"My nails have small dents or pits in them, and..."


In [13]:
df['label'].value_counts()

label
Psoriasis                          50
Varicose Veins                     50
Typhoid                            50
Chicken pox                        50
Impetigo                           50
Dengue                             50
Fungal infection                   50
Common Cold                        50
Pneumonia                          50
Dimorphic Hemorrhoids              50
Arthritis                          50
Acne                               50
Bronchial Asthma                   50
Hypertension                       50
Migraine                           50
Cervical spondylosis               50
Jaundice                           50
Malaria                            50
urinary tract infection            50
allergy                            50
gastroesophageal reflux disease    50
drug reaction                      50
peptic ulcer disease               50
diabetes                           50
Name: count, dtype: int64

In [14]:
def tag_to_wordnet(tag: str):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    if tag.startswith('N'):
        return wordnet.NOUN
    return wordnet.NOUN

def preprocess(sentences: str):
    tokens = word_tokenize(sentences)
    cleaned_tokens = [token.lower() for token in tokens if token.lower() not in stopwords.words('english') and token.lower() not in punctuation]
    tagged_tokens = pos_tag(cleaned_tokens)
    stemmed_tokens = [WordNetLemmatizer().lemmatize(token, tag_to_wordnet(tag)) for token, tag in tagged_tokens]
    return stemmed_tokens

def feature_extraction(tokens: list[str], fd: FreqDist):
    dict_feature = {}
    for token in fd.keys():
        dict_feature[token] = (token in tokens)
    return dict_feature


In [15]:
sentences = "  ".join(df['text'])
print(sentences)

cleaned_tokens = preprocess(sentences)
print(cleaned_tokens)
fd = FreqDist(cleaned_tokens)

X = df['text']
y = df['label']

I have been experiencing a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.  My skin has been peeling, especially on my knees, elbows, and scalp. This peeling is often accompanied by a burning or stinging sensation.  I have been experiencing joint pain in my fingers, wrists, and knees. The pain is often achy and throbbing, and it gets worse when I move my joints.  There is a silver like dusting on my skin, especially on my lower back and scalp. This dusting is made up of small scales that flake off easily when I scratch them.  My nails have small dents or pits in them, and they often feel inflammatory and tender to the touch. Even there are minor rashes on my arms.  The skin on my palms and soles is thickened and has deep cracks. These cracks are painful and bleed easily.  The skin around my mouth, nose, and eyes is red and inflamed. It is often itchy and uncomfortable. There is a noticeable inflammation in my nails.  My

In [16]:
def train_model():
    all_statement = [(feature_extraction(preprocess(statement), fd), sentiment) for statement, sentiment in zip(X, y)]
    train_data, test_data = train_test_split(all_statement, test_size=0.2, random_state=42)
    classifier = NaiveBayesClassifier.train(train_data)
    print(classifier.show_most_informative_features(5))
    print(accuracy(classifier, test_data))
    return classifier
    

In [17]:
model = None

def load_model():
    global model
    try:
        with open("model.pkl", "rb") as f:
            model = pickle.load(f)
            print("Model Exists! Using current model")
    except:
        print("Model doesn't exist! Training model...")
        model = train_model()
        save_model(model)

def save_model(model):
    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)


In [18]:
load_model()

Model doesn't exist! Training model...
Most Informative Features
                    neck = True           Cervic : Psoria =     29.7 : 1.0
                    back = False            Acne : Cervic =     29.0 : 1.0
                    rash = False          Cervic : Acne   =     29.0 : 1.0
                    skin = False          Cervic : Acne   =     29.0 : 1.0
                    high = False          drug r : Malari =     26.4 : 1.0
None
0.9666666666666667
